# Crawling Berita Kompas

In [1]:
import requests, uuid, time, random
from requests.adapters import HTTPAdapter, Retry
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urlunparse
import pandas as pd
import os

# Header untuk menyamarkan request agar dianggap seperti browser biasa
DEFAULT_UA = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36"
)
headers = {"User-Agent": DEFAULT_UA, "Accept-Language": "id,en;q=0.9"}

# Session dengan retry & backoff
session = requests.Session()
retries = Retry(total=3, backoff_factor=0.6, status_forcelist=[429, 500, 502, 503, 504], allowed_methods=["GET"])
adapter = HTTPAdapter(max_retries=retries, pool_connections=10, pool_maxsize=20)
session.mount("http://", adapter)
session.mount("https://", adapter)


# Berita yang akan di crawling

In [2]:
# URL awal situs (beranda Kompas)
START_URL = "https://www.kompas.com/"


# Fungsi Utility: Normalisasi URL & Ambil Teks

In [3]:
def canonicalize(url):
    # Normalisasi URL supaya tidak ada parameter yang bikin duplikat
    p = urlparse(url)
    return urlunparse(("https", p.netloc, p.path.rstrip('/'), "", "", ""))

def is_valid_article_url(url: str) -> bool:
    # Batasi hanya artikel teks, exclude video/galeri/dll
    lower = url.lower()
    if not ("kompas.com" in lower):
        return False
    if any(x in lower for x in ["video.kompas.com", "/galeri/", "/watch/", "/foto/", "/player_", "/live/", "kgnow.com"]):
        return False
    return "/read/" in lower

def first_text(soup, selectors):
    # Ambil teks pertama yang cocok dari list CSS selector
    for sel in selectors:
        nodes = soup.select(sel)
        if nodes:
            return " ".join([n.get_text(strip=True) for n in nodes])
    return None

KNOWN_CATEGORIES = {
    "nasional", "global", "megapolitan", "regional", "pemilu", "hype",
    "tekno", "money", "otomotif", "bola", "entertainment", "lifestyle",
    "travel", "properti", "sains", "health", "edukasi", "tren", "lestari"
}

# Batasi hanya 9 kategori yang diizinkan untuk dikumpulkan
ALLOWED_CATEGORIES = {
    "Nasional", "Global", "Megapolitan", "Regional",
    "Tekno", "Money", "Otomotif", "Bola", "Entertainment"
}

def guess_category_from_url(url: str) -> str:
    p = urlparse(url)
    host = p.netloc.split(".")
    sub = host[0] if len(host) > 2 else host[0]  # e.g., tekno.kompas.com
    if sub and sub not in ("www", "kompas") and sub in KNOWN_CATEGORIES:
        return sub.capitalize()
    # fallback from first path segment
    path = p.path.strip("/").split("/")
    if path:
        seg = path[0].lower()
        if seg in KNOWN_CATEGORIES:
            return seg.capitalize()
    return "Kompas"


# Variabel untuk Cegah Duplikat

In [4]:
seen_urls = set()

# Fungsi Parsing Detail Berita

In [5]:
def parse_news_detail(url, default_category):
    try:
        # Request detail berita
        res = session.get(url, headers=headers, timeout=10)
        if res.status_code != 200:
            print(f"[ERROR] status {res.status_code} for {url}")
            return None
        soup = BeautifulSoup(res.text, "html.parser")

        # Ambil canonical URL
        canon_tag = soup.select_one('link[rel="canonical"]')
        final_url = canonicalize(canon_tag['href'] if canon_tag and canon_tag.get('href') else res.url)
        url_key = final_url
        if url_key in seen_urls:
            return None
        seen_urls.add(url_key)

        # Judul berita
        title = first_text(soup, ["h1.read__title", "h1.article__title", "h1.title", "h1.read__title__text"])
        if not title:
            m = soup.find('meta', property='og:title') or soup.find('meta', attrs={'name': 'title'})
            title = m['content'] if m and m.get('content') else ""

        # Isi berita
        content = first_text(soup, [
            "div.read__content p",
            "div.article__content p",
            "div.detail_text p",
            "div.article__lead p",
            "div._article_content p",
            "article p"
        ])
        if not content:
            m = soup.find('meta', property='og:description') or soup.find('meta', attrs={'name': 'description'})
            content = m['content'] if m and m.get('content') else ""

        # Kategori (ambil breadcrumb jika ada, fallback ke default_category)
        breadcrumb = [a.get_text(strip=True) for a in soup.select("a.breadcrumb__link, span.breadcrumb__link, nav.breadcrumb a")]
        detected_kategori = breadcrumb[-1] if breadcrumb else default_category

        # Tanggal publikasi (jika ada)
        pub = first_text(soup, ["div.read__time", "div.article__date", "time.read__time"])
        if not pub:
            m = soup.find('meta', property='article:published_time') or soup.find('meta', attrs={'name': 'date'})
            pub = m['content'] if m and m.get('content') else ""

        return {
            "id": str(uuid.uuid4()),        # ID unik
            "judul": title,                 # Judul berita
            "isi": content,                 # Isi berita
            "kategori": default_category,   # Kategori awal (hardcode)
            "detected_kategori": detected_kategori, # Kategori dari breadcrumb
            "tanggal": pub,                 # Tanggal publikasi jika tersedia
            "url": final_url                # URL canonical
        }
    except Exception as e:
        print(f"[ERROR parsing] {url}: {e}")
        return None


# Fungsi Crawling per Kategori

In [6]:
def same_domain(url: str, base_host: str) -> bool:
    p = urlparse(url)
    return p.netloc.endswith(base_host)


def crawl_site(start_url: str, max_pages: int = 2000, max_articles: int = 10000):
    base_host = urlparse(start_url).netloc
    print(f"[CRAWL SITE] {start_url} (host: {base_host})")

    to_visit = [canonicalize(start_url)]
    visited_pages = set()
    collected = []
    per_category_counts = {}  # kategori -> jumlah terkumpul

    while to_visit and len(visited_pages) < max_pages and len(collected) < max_articles:
        current = to_visit.pop(0)
        if current in visited_pages:
            continue
        visited_pages.add(current)

        try:
            res = session.get(current, headers=headers, timeout=10)
            if res.status_code != 200:
                print(f"  [SKIP] status {res.status_code} {current}")
                continue
            soup = BeautifulSoup(res.text, "html.parser")
        except Exception as e:
            print(f"  [ERR] {current}: {e}")
            continue

        # Kumpulkan link baru (same domain)
        anchors = soup.select("a[href]")
        page_links = []
        for a in anchors:
            href = a.get("href")
            if not href:
                continue
            full = urljoin(current, href)
            full = canonicalize(full)
            if not same_domain(full, base_host):
                continue
            # hanya antrekan halaman navigasi/listing; artikel di-parse langsung di bawah
            if full not in visited_pages and full not in to_visit:
                page_links.append(full)

        # Parse artikel teks pada halaman ini (baik halaman list maupun beranda)
        article_links = []
        for a in anchors:
            href = a.get("href")
            if not href:
                continue
            full = urljoin(current, href)
            if not is_valid_article_url(full):
                continue
            key = canonicalize(full)
            if key in seen_urls:
                continue
            article_links.append(key)

        if article_links:
            print(f"  [PAGE] {current} → {len(article_links)} article links")

        for link in article_links:
            if len(collected) >= max_articles:
                break
            cat = guess_category_from_url(link)
            if cat not in ALLOWED_CATEGORIES:
                continue
            if per_category_counts.get(cat, 0) >= 200:
                continue
            print(f"    - Fetching: {link} [{cat}]")
            news = parse_news_detail(link, default_category=cat)
            if news and news.get("kategori"):
                collected.append(news)
                per_category_counts[cat] = per_category_counts.get(cat, 0) + 1
            time.sleep(0.6 + random.random()*0.7)

        # Tambahkan link navigasi untuk BFS (dibatasi untuk menghindari ledakan)
        random.shuffle(page_links)
        for nxt in page_links[:200]:  # batasi antrian per halaman
            if len(visited_pages) + len(to_visit) >= max_pages:
                break
            to_visit.append(nxt)

        # jeda kecil antar halaman
        time.sleep(0.4 + random.random()*0.6)

    return collected


# Hasil Crawling

In [7]:
if __name__ == "__main__":
    output_path = "hasil_crawling_kompas.csv"
    existing = set()
    if os.path.exists(output_path):
        try:
            prev = pd.read_csv(output_path)
            for u in prev.get("url", []):
                existing.add(canonicalize(str(u)))
        except Exception:
            pass

    total = 0
    batch = crawl_site(START_URL, max_pages=3000, max_articles=20000)
    if batch:
        filtered = [x for x in batch if canonicalize(x.get("url", "")) not in existing]
        if filtered:
            df = pd.DataFrame(filtered)
            if os.path.exists(output_path):
                df.to_csv(output_path, mode="a", header=False, index=False, encoding="utf-8-sig")
            else:
                df.to_csv(output_path, index=False, encoding="utf-8-sig")
            for x in filtered:
                existing.add(canonicalize(x.get("url", "")))
            total += len(filtered)
            print(f"[WRITE] {len(filtered)} rows appended from site crawl")


    print(f"[DONE] Total appended {total} berita ke {output_path}")


[CRAWL SITE] https://www.kompas.com/ (host: www.kompas.com)


  [PAGE] https://www.kompas.com → 80 article links
    - Fetching: https://bola.kompas.com/read/2025/10/09/22112868/keluarga-pemain-timnas-indonesia-dapat-perlakuan-tak-menyenangkan-di-jeddah [Bola]


    - Fetching: https://megapolitan.kompas.com/read/2025/10/09/21292211/dapat-penangguhan-penahanan-tiktokers-figha-lesmana-minta-maaf [Megapolitan]


    - Fetching: https://money.kompas.com/read/2025/10/09/220815126/toyota-komentari-soal-penerapan-etanol-10-persen-pada-bbm [Money]


    - Fetching: https://nasional.kompas.com/read/2025/10/09/21255531/yusril-sebut-hambali-akan-diadili-di-pengadilan-militer-as-november-2025 [Nasional]


...

    - Fetching: https://regional.kompas.com/read/2025/10/09/225232578/massa-rusak-kantor-dpd-golkar-maluku-diduga-terkait-masalah-paw [Regional]


    - Fetching: https://megapolitan.kompas.com/read/2025/10/09/22491331/jasa-teman-jalan-jakarta-kian-diminati-sudah-layani-wisatawan-dari [Megapolitan]


    - Fetching: https://megapolitan.kompas.com/read/2025/10/09/22473391/perbaikan-jembatan-plawad-2-bikin-macet-waktu-tempuh-bisa-1-jam [Megapolitan]


    - Fetching: https://tekno.kompas.com/read/2025/10/09/22450027/link-dan-cara-po-iphone-17-pro-di-indonesia-dan-daftar-harganya [Tekno]


KeyboardInterrupt: 